# Session 2: LangChain Development & Tool Integration (Student Notebook)

Welcome to Session 2! In this notebook, you will learn LangChain — the most popular framework for building LLM applications. You will master its core abstractions, compose chains using LCEL, create custom tools, and build a simple RAG pipeline.

## Learning Objectives

By the end of this session, you will be able to:

1. **Use ChatModels and PromptTemplates** as LangChain's core building blocks
2. **Compose chains with LCEL** using the pipe operator (`|`)
3. **Create custom tools** using the `@tool` decorator
4. **Load and split documents** for retrieval workflows
5. **Build a RAG pipeline** that grounds LLM answers in external knowledge
6. **Manage conversation memory** across multi-turn interactions

In [ ]:
# ============================================================
# Imports and Setup
# ============================================================

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.tools import tool
from langchain_core.documents import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
import json
import os

# Ensure your OpenAI API key is set:
#   export OPENAI_API_KEY="sk-..."

print("All imports successful!")

---
## Demos (Follow Along)

The following five demos are fully coded. Run each cell, observe the output, and make sure you understand what is happening before moving on.

### Demo 1: LangChain Basics — ChatModels and PromptTemplates

ChatModels wrap LLM APIs into a unified interface. PromptTemplates let you parameterize prompts with variables so they become reusable across different inputs.

In [ ]:
# Demo 1 - LangChain Basics: ChatModels and PromptTemplates

# Step 1: Initialize a ChatModel
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

# Step 2: Simple invocation with a string
response = llm.invoke("What is LangChain in one sentence?")
print("Simple invocation:")
print(f"  Type: {type(response)}")
print(f"  Content: {response.content}")

# Step 3: Invocation with message objects
messages = [
    SystemMessage(content="You are a Python expert. Be concise."),
    HumanMessage(content="What is a list comprehension?")
]
response = llm.invoke(messages)
print(f"\nWith messages:\n  {response.content}")

# Step 4: PromptTemplates for reusable prompts
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert in {domain}. Answer in {style} style."),
    ("human", "{question}")
])

formatted = prompt.format_messages(
    domain="machine learning",
    style="casual",
    question="What is gradient descent?"
)
print(f"\nFormatted prompt messages: {len(formatted)}")
for msg in formatted:
    print(f"  [{msg.type}]: {msg.content[:80]}...")

response = llm.invoke(formatted)
print(f"\nResponse: {response.content[:200]}...")

### Demo 2: Building Chains with LCEL (Pipe Operator)

LCEL (LangChain Expression Language) uses the `|` operator to compose components into chains. The pattern is: `prompt | model | parser`. Each component implements the Runnable interface, so they can be freely combined.

In [ ]:
# Demo 2 - Building Chains with LCEL

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.5)

# Step 1: Simple chain: prompt -> LLM -> string output
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that explains concepts clearly."),
    ("human", "Explain {concept} in exactly {num_sentences} sentences.")
])

chain = prompt | llm | StrOutputParser()

result = chain.invoke({"concept": "recursion", "num_sentences": "3"})
print("Simple chain result:")
print(result)

# Step 2: Chain with JSON output
json_prompt = ChatPromptTemplate.from_messages([
    ("system", "Output a JSON object with keys: definition, example, difficulty (1-10)."),
    ("human", "Describe the programming concept: {concept}")
])

json_chain = json_prompt | llm | JsonOutputParser()

result = json_chain.invoke({"concept": "decorators"})
print(f"\nJSON chain result (type={type(result).__name__}):")
print(json.dumps(result, indent=2))

# Step 3: Multi-step chains
summarize_prompt = ChatPromptTemplate.from_template("Summarize this in one sentence: {text}")
translate_prompt = ChatPromptTemplate.from_template("Translate this to French: {text}")

summary_chain = summarize_prompt | llm | StrOutputParser()
translate_chain = translate_prompt | llm | StrOutputParser()

text = "Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. It focuses on developing algorithms that can access data and use it to learn for themselves."

summary = summary_chain.invoke({"text": text})
print(f"\nSummary: {summary}")

translation = translate_chain.invoke({"text": summary})
print(f"French: {translation}")

### Demo 3: Creating and Using Custom Tools

Tools let LLMs interact with the real world. In LangChain, use the `@tool` decorator to turn any Python function into a tool that an LLM can discover and invoke. The docstring becomes the tool's description.

In [ ]:
# Demo 3 - Creating and Using Custom Tools

# Step 1: Define custom tools
@tool
def word_count(text: str) -> int:
    """Count the number of words in a text string."""
    return len(text.split())

@tool
def reverse_string(text: str) -> str:
    """Reverse a given text string."""
    return text[::-1]

@tool
def calculate_average(numbers: list[float]) -> float:
    """Calculate the average of a list of numbers."""
    if not numbers:
        return 0.0
    return sum(numbers) / len(numbers)

# Step 2: Inspect tool metadata
print("Tool: word_count")
print(f"  Name: {word_count.name}")
print(f"  Description: {word_count.description}")

# Step 3: Invoke tools directly
print(f"\nword_count('Hello world from LangChain'): {word_count.invoke('Hello world from LangChain')}")
print(f"reverse_string('LangChain'): {reverse_string.invoke('LangChain')}")
print(f"calculate_average([10, 20, 30]): {calculate_average.invoke([10, 20, 30])}")

# Step 4: Bind tools to an LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
llm_with_tools = llm.bind_tools([word_count, reverse_string, calculate_average])

response = llm_with_tools.invoke("How many words are in the sentence 'The quick brown fox jumps'?")
print(f"\nLLM response with tools:")
print(f"  Content: {response.content}")
print(f"  Tool calls: {response.tool_calls}")

### Demo 4: Document Loading and Text Splitting

RAG starts with loading documents and splitting them into manageable chunks. The `RecursiveCharacterTextSplitter` splits at natural boundaries (paragraphs, sentences) and maintains overlap between chunks so context is not lost at boundaries.

In [ ]:
# Demo 4 - Document Loading and Text Splitting

# Step 1: Create sample documents
documents = [
    Document(
        page_content="""Large Language Models (LLMs) are neural networks trained on massive text datasets.
They use the Transformer architecture, which relies on self-attention mechanisms to process
sequences of tokens. Modern LLMs like GPT-4 and Claude have billions of parameters and can
perform a wide range of natural language tasks including translation, summarization, question
answering, and code generation.""",
        metadata={"source": "llm_overview.txt", "chapter": 1}
    ),
    Document(
        page_content="""Retrieval-Augmented Generation (RAG) is a technique that combines LLMs with
external knowledge retrieval. Instead of relying solely on the model's training data, RAG
systems retrieve relevant documents from a knowledge base and include them in the prompt
context. This approach reduces hallucinations and keeps information up-to-date.""",
        metadata={"source": "rag_guide.txt", "chapter": 2}
    )
]

print(f"Loaded {len(documents)} documents")
for doc in documents:
    print(f"  Source: {doc.metadata['source']} | Length: {len(doc.page_content)} chars")

# Step 2: Split documents into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=30,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = splitter.split_documents(documents)
print(f"\nSplit into {len(chunks)} chunks:")
for i, chunk in enumerate(chunks):
    print(f"\n  Chunk {i+1} ({len(chunk.page_content)} chars) [from {chunk.metadata['source']}]:")
    print(f"    {chunk.page_content[:100]}...")

# Step 3: Show how overlap works
print("\nOverlap demonstration:")
print(f"  End of chunk 1  : ...{chunks[0].page_content[-40:]}")
if len(chunks) > 1:
    print(f"  Start of chunk 2: {chunks[1].page_content[:40]}...")

### Demo 5: Building a Simple RAG Pipeline

This demo puts it all together: a knowledge base of text chunks, a simple retrieval function, and an LCEL chain that generates answers grounded in retrieved context.

In [ ]:
# Demo 5 - Building a Simple RAG Pipeline

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Step 1: Knowledge base
knowledge_base = [
    "LangChain is a framework for building LLM applications. It provides composable components for prompts, models, and chains.",
    "LCEL (LangChain Expression Language) uses the pipe operator (|) to chain components. The pattern is: prompt | model | parser.",
    "LangGraph extends LangChain with graph-based workflows. It supports cycles, conditional edges, and persistent state.",
    "RAG (Retrieval-Augmented Generation) improves LLM accuracy by retrieving relevant documents before generating answers.",
    "Tools in LangChain are defined using the @tool decorator. They allow LLMs to interact with external systems and APIs.",
    "Memory in LangChain stores conversation history. Common types include ConversationBufferMemory and ConversationSummaryMemory."
]

# Step 2: Simple keyword-based retrieval
def simple_retrieve(query, k=2):
    scored = []
    query_words = set(query.lower().split())
    for chunk in knowledge_base:
        chunk_words = set(chunk.lower().split())
        overlap = len(query_words & chunk_words)
        scored.append((overlap, chunk))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [chunk for _, chunk in scored[:k]]

# Step 3: RAG chain
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the question based ONLY on the provided context. If the context does not contain the answer, say 'I don't have enough information.'\n\nContext:\n{context}"),
    ("human", "{question}")
])

rag_chain = rag_prompt | llm | StrOutputParser()

# Step 4: Run queries
questions = [
    "What is LCEL and how does it work?",
    "How does RAG improve LLM accuracy?",
    "What is quantum computing?"
]

for question in questions:
    retrieved = simple_retrieve(question)
    context = "\n".join(f"- {chunk}" for chunk in retrieved)
    answer = rag_chain.invoke({"context": context, "question": question})
    print(f"Q: {question}")
    print(f"A: {answer}\n")

---
## Tasks (Your Turn!)

Now it is your turn to practice. Each task below has a description, hints, and a code skeleton with `TODO` placeholders. Fill in the code to complete each task.

### Task 1: Build a Chain with Prompt Templates and Output Parsers

Create an LCEL chain that takes a topic and difficulty level, generates a quiz question with four options, and returns the result as parsed JSON.

In [ ]:
# Task 1 - Build a Chain with Prompt Templates and Output Parsers

# TODO: Create an LCEL chain that:
# 1. Takes a topic and difficulty level as inputs
# 2. Uses a PromptTemplate to generate a quiz question about the topic
# 3. Parses the output as JSON with fields: question, options (list of 4), correct_answer
#
# Hint: Use ChatPromptTemplate.from_messages() with system and human messages
# Hint: Tell the system to output JSON in the system message
# Hint: Use JsonOutputParser() as the final step in the chain

def create_quiz_chain():
    """Create a chain that generates quiz questions."""
    # YOUR CODE HERE
    pass


# Test your chain
# chain = create_quiz_chain()
# result = chain.invoke({"topic": "Python programming", "difficulty": "intermediate"})
# print(json.dumps(result, indent=2))

### Task 2: Create Custom Tools for Math and Text Processing

Create three custom tools using the `@tool` decorator: a Fibonacci generator, a text statistics calculator, and a Caesar cipher encoder. Then bind them to an LLM.

In [ ]:
# Task 2 - Create Custom Tools for Math and Text Processing

# TODO: Create three custom tools using the @tool decorator:
# 1. fibonacci(n: int) -> list - returns first n Fibonacci numbers
# 2. text_stats(text: str) -> dict - returns word count, char count, sentence count
# 3. caesar_cipher(text: str, shift: int) -> str - encrypts text with Caesar cipher
#
# Then bind all tools to an LLM and test tool invocation.
#
# Hint: Use the @tool decorator from langchain_core.tools
# Hint: Include clear docstrings -- LangChain uses them as tool descriptions
# Hint: Use llm.bind_tools([...]) to attach tools to the model

# YOUR CODE HERE


# Test your tools directly
# print(fibonacci.invoke(8))
# print(text_stats.invoke("Hello world. How are you?"))
# print(caesar_cipher.invoke({"text": "hello", "shift": 3}))

### Task 3: Implement a Conversational Chain with Memory

Build a conversational chain that maintains message history across turns, so the LLM remembers what was discussed earlier in the conversation.

In [ ]:
# Task 3 - Implement a Conversational Chain with Memory

# TODO: Build a conversational chain that:
# 1. Maintains a list of previous messages (chat history)
# 2. Uses MessagesPlaceholder in the prompt to include history
# 3. Has a chat() method that appends user/assistant messages to history
# 4. Returns the assistant's response
#
# Hint: Use ChatPromptTemplate with MessagesPlaceholder("history")
# Hint: Store history as a list of HumanMessage and AIMessage objects
# Hint: Pass history to chain.invoke({"history": self.history, "input": user_input})

class ConversationalChain:
    def __init__(self):
        """Initialize the chain with memory."""
        # YOUR CODE HERE
        pass

    def chat(self, user_input):
        """Send a message and get a response, maintaining history."""
        # YOUR CODE HERE
        pass

    def get_history(self):
        """Return the conversation history."""
        # YOUR CODE HERE
        pass


# Test your chain
# conv = ConversationalChain()
# print(conv.chat("My name is Alex. I'm learning LangChain."))
# print(conv.chat("What's my name and what am I learning?"))
# print(f"History: {len(conv.get_history())} messages")

### Task 4: Build a RAG-Powered Q&A System

Build a complete RAG system that takes documents, splits them into chunks, retrieves relevant chunks for a query, and generates grounded answers.

In [ ]:
# Task 4 - Build a RAG-Powered Q&A System

# TODO: Build a complete RAG system that:
# 1. Takes a list of text documents as input
# 2. Splits them into chunks using RecursiveCharacterTextSplitter
# 3. Implements a retrieve() method that finds relevant chunks for a query
# 4. Implements an ask() method that retrieves context and generates an answer
# 5. Includes the source metadata in the response
#
# Hint: Split documents with chunk_size=300, chunk_overlap=50
# Hint: Use keyword matching for retrieval (or implement simple TF-IDF)
# Hint: Include source information in the prompt context

class SimpleRAG:
    def __init__(self, documents):
        """Initialize with a list of Document objects, split into chunks."""
        # YOUR CODE HERE
        pass

    def retrieve(self, query, k=3):
        """Retrieve the top-k most relevant chunks for a query."""
        # YOUR CODE HERE
        pass

    def ask(self, question):
        """Answer a question using retrieved context."""
        # YOUR CODE HERE
        pass


# Test your RAG system
# docs = [
#     Document(page_content="Python was created by Guido van Rossum and first released in 1991. It emphasizes code readability with significant whitespace.", metadata={"source": "python.txt"}),
#     Document(page_content="LangChain provides composable components for building LLM applications. It includes models, prompts, chains, and tools.", metadata={"source": "langchain.txt"}),
# ]
# rag = SimpleRAG(docs)
# print(rag.ask("Who created Python?"))

---
## Summary

In this session you learned the core LangChain building blocks for LLM application development:

1. **ChatModels & PromptTemplates** -- How LangChain wraps LLM APIs and provides reusable, parameterized prompt templates.
2. **LCEL Chains** -- How the pipe operator (`|`) composes prompt, model, and parser into clean, readable chains.
3. **Custom Tools** -- How the `@tool` decorator turns Python functions into tools that LLMs can discover and invoke.
4. **Document Loading & Splitting** -- How to prepare text for retrieval by splitting it into overlapping chunks.
5. **RAG Pipelines** -- How to combine retrieval with generation to ground LLM answers in external knowledge.

**Up Next:** In Session 3, we will move from linear chains to graph-based workflows with LangGraph — enabling conditional routing, cycles, and the complex control flows that real-world agents require.